In [ ]:
# databricks-connect automatically adds the project root to the PYTHONPATH, so we can import modules from src
# But when running this script on Databricks Workspace, we need to add the project root to the PYTHONPATH manually
# import os
# import sys

# current_dir = os.getcwd()
# parent_dir = os.path.abspath(os.path.join(current_dir, "..", "..", ".."))
# sys.path.append(parent_dir)

In [ ]:
from citibike.citibike_utils import get_trip_duration_mins
from utils.datetime_utils import timestamp_to_date_col

from pyspark.sql.functions import create_map, lit

In [ ]:
pipeline_id = dbutils.widgets.get("pipeline_id")
run_id = dbutils.widgets.get("run_id")
task_id = dbutils.widgets.get("task_id")
processed_timestamp = dbutils.widgets.get("processed_timestamp")

catalog = dbutils.widgets.get("catalog")

In [ ]:
df = spark.read.table(f"{catalog}.01_bronze.jc_citibike")

In [ ]:
# display(df.limit(5))

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,metadata
0,29DAF43DD84B4B7A,electric_bike,2025-03-20 18:58:31.217,2025-03-20 19:00:46.466,6 St & Grand St,HB302,Mama Johnson Field - 4 St & Jackson St,HB404,41,-74,41,-74,member,"{'pipeline_id': 'placeholder', 'run_id': 'placeholder', 'task_id': 'placeholder', 'processed_timestamp': 'placeholder'}"
1,B11B4220F7195025,electric_bike,2025-03-29 11:01:25.124,2025-03-29 11:11:09.383,Heights Elevator,JC059,Jersey & 3rd,JC074,41,-74,41,-74,member,"{'pipeline_id': 'placeholder', 'run_id': 'placeholder', 'task_id': 'placeholder', 'processed_timestamp': 'placeholder'}"
2,18D5B30305F602B9,electric_bike,2025-03-01 16:05:32.346,2025-03-01 16:07:43.156,Jersey & 3rd,JC074,Hamilton Park,JC009,41,-74,41,-74,member,"{'pipeline_id': 'placeholder', 'run_id': 'placeholder', 'task_id': 'placeholder', 'processed_timestamp': 'placeholder'}"
3,532EB2D9DB68567D,electric_bike,2025-03-21 18:44:15.137,2025-03-21 18:51:00.763,Jersey & 3rd,JC074,Jersey & 6th St,JC027,41,-74,41,-74,member,"{'pipeline_id': 'placeholder', 'run_id': 'placeholder', 'task_id': 'placeholder', 'processed_timestamp': 'placeholder'}"
4,EA7C9C945D7D57AA,electric_bike,2025-03-20 11:08:27.226,2025-03-20 11:12:28.545,6 St & Grand St,HB302,Madison St & 1 St,HB402,41,-74,41,-74,member,"{'pipeline_id': 'placeholder', 'run_id': 'placeholder', 'task_id': 'placeholder', 'processed_timestamp': 'placeholder'}"


In [ ]:
df = get_trip_duration_mins(spark, df, "started_at", "ended_at", "trip_duration_mins")

In [ ]:
# display(df.select("started_at", "ended_at", "trip_duration_mins").limit(5))

,started_at,ended_at,trip_duration_mins
0,2025-03-20 18:58:31.217,2025-03-20 19:00:46.466,2.250000
1,2025-03-29 11:01:25.124,2025-03-29 11:11:09.383,9.733333
2,2025-03-01 16:05:32.346,2025-03-01 16:07:43.156,2.183333
3,2025-03-21 18:44:15.137,2025-03-21 18:51:00.763,6.750000
4,2025-03-20 11:08:27.226,2025-03-20 11:12:28.545,4.016667


In [10]:
df = timestamp_to_date_col(spark, df, "started_at", "trip_start_date")

In [ ]:
# display(df.select("started_at", "trip_start_date").limit(5))

,started_at,trip_start_date
0,2025-03-20 18:58:31.217,2025-03-20
1,2025-03-29 11:01:25.124,2025-03-29
2,2025-03-01 16:05:32.346,2025-03-01
3,2025-03-21 18:44:15.137,2025-03-21
4,2025-03-20 11:08:27.226,2025-03-20


In [ ]:
df = df.withColumn(
    "metadata",
    create_map(
        lit("pipeline_id"),
        lit(pipeline_id),
        lit("run_id"),
        lit(run_id),
        lit("task_id"),
        lit(task_id),
        lit("processed_timestamp"),
        lit(processed_timestamp),
    ),
)

In [13]:
df = df.select(
    "ride_id",
    "trip_start_date",
    "started_at",
    "ended_at",
    "start_station_name",
    "end_station_name",
    "trip_duration_mins",
    "metadata"
    )

In [ ]:
df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{catalog}.02_silver.jc_citibike"
)